In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# הזנה קדמית קלאסית ובקרת זרימה (מעגלים דינמיים)

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



# הזנה קדמית קלאסית ובקרת זרימה


<details>
<summary><b>גרסאות חבילות</b></summary>

הקוד בדף זה פותח תוך שימוש בדרישות הבאות.
אנחנו ממליצים להשתמש בגרסאות אלו או בגרסאות חדשות יותר.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
> **Note:** הגרסה החדשה של מעגלים דינמיים זמינה כעת לכל המשתמשים בכל ה-Backends. כעת תוכל להריץ מעגלים דינמיים בקנה מידה שימושי. ראה [את ההכרזה](/announcements/product-updates/2025-09-25-new-dynamic-circuits) לפרטים נוספים.

מעגלים דינמיים הם כלים רבי עוצמה שבהם אפשר למדוד Qubits באמצע ביצוע מעגל קוונטי, ואז לבצע פעולות לוגיקה קלאסית בתוך המעגל, בהתבסס על תוצאות אותן מדידות אמצע-מעגל. תהליך זה ידוע גם בשם _הזנה קדמית קלאסית_. למרות שאלו עדיין ימיה הראשונים של ההבנה כיצד לנצל בצורה הטובה ביותר את המעגלים הדינמיים, קהילת המחקר הקוונטי כבר זיהתה מספר מקרי שימוש, כגון הבאים:

* הכנת מצב קוונטי יעילה, כגון [מצב GHZ,](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) [מצב W,](https://arxiv.org/abs/2403.07604) (למידע נוסף על מצב W, ראה גם ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)) ומחלקה רחבה של [מצבי מכפלת מטריצה](https://arxiv.org/abs/2404.16083)
* [שזירה ארוכת טווח יעילה](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) בין Qubits על אותו שבב באמצעות מעגלים רדודים
* [דגימה יעילה של מעגלים דמויי IQP](https://arxiv.org/pdf/2505.04705)

השיפורים שמביאים המעגלים הדינמיים, לעומת זאת, כרוכים בפשרות. מדידות אמצע-מעגל ופעולות קלאסיות בדרך כלל דורשות זמן ביצוע ארוך יותר מ-Gates דו-Qubit, ועלייה זו בזמן עשויה לבטל את היתרונות של עומק מעגל מצומצם. לכן, קיצור משך מדידת אמצע-המעגל הוא תחום מיקוד לשיפור כאשר IBM Quantum&reg; משחררת את [הגרסה החדשה](/announcements/product-updates/2025-03-03-new-version-dynamic-circuits) של מעגלים דינמיים.

[מפרט OpenQASM 3](https://openqasm.com/language/classical.html#looping-and-branching) מגדיר מספר מבני בקרת זרימה, אך Qiskit Runtime תומך כרגע רק במשפט `if` המותנה. ב-Qiskit SDK, זה מתאים לשיטת [if_test](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test) ב-[QuantumCircuit.](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) שיטה זו מחזירה [מנהל הקשר](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) ומשמשת בדרך כלל במשפט `with`. מדריך זה מתאר כיצד להשתמש במשפט מותנה זה.

> **Note:** דוגמאות הקוד במדריך זה משתמשות בהוראת המדידה הסטנדרטית למדידות אמצע-מעגל. עם זאת, מומלץ להשתמש בהוראת [`MidCircuitMeasure`](/guides/measure-qubits#midcircuit) במקום, אם ה-Backend תומך בה. ראה את [תיעוד מדידות אמצע-מעגל](/guides/measure-qubits#mid-circuit-measurements) לפרטים.
## משפט `if`
משפט ה-`if` משמש לביצוע פעולות באופן מותנה בהתבסס על ערכו של סיבית קלאסי או רגיסטר.

בדוגמה הבאה, אנחנו מיישמים Gate הדמארד על Qubit ומודדים אותו. אם התוצאה היא 1, אז אנחנו מיישמים Gate X על ה-Qubit, מה שהאפקט שלו הוא היפוכו בחזרה למצב 0. לאחר מכן אנחנו מודדים את ה-Qubit שוב. תוצאת המדידה המתקבלת צריכה להיות 0 עם הסתברות של 100%.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg)

ניתן לתת למשפט ה-`with` יעד השמה שהוא עצמו מנהל הקשר שניתן לאחסן ולהשתמש בו לאחר מכן ליצירת בלוק else, אשר מבוצע בכל פעם שתוכן בלוק ה-`if` *אינו* מבוצע.

בדוגמה הבאה, אנחנו מאתחלים רגיסטרים עם שני Qubits ושני סיביות קלאסיות. אנחנו מיישמים Gate הדמארד על ה-Qubit הראשון ומודדים אותו. אם התוצאה היא 1, אז אנחנו מיישמים Gate הדמארד על ה-Qubit השני; אחרת, אנחנו מיישמים Gate X על ה-Qubit השני. לבסוף, אנחנו מודדים גם את ה-Qubit השני.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg)

בנוסף לתנאי על סיבית קלאסי בודד, ניתן גם להתנות על ערכו של רגיסטר קלאסי המורכב ממספר סיביות.

בדוגמה הבאה, אנחנו מיישמים Gates הדמארד על שני Qubits ומודדים אותם. אם התוצאה היא `01`, כלומר ה-Qubit הראשון הוא 1 וה-Qubit השני הוא 0, אז אנחנו מיישמים Gate X על Qubit שלישי. לבסוף, אנחנו מודדים את ה-Qubit השלישי. שים לב שלשם הבהירות, בחרנו לציין את מצב הסיבית הקלאסית השלישית, שהוא 0, בתנאי ה-`if`. בציור המעגל, התנאי מסומן על ידי עיגולים על הסיביות הקלאסיות שמתנים עליהן. עיגול שחור מציין תנאי על 1, בעוד שעיגול לבן מציין תנאי על 0.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg)

## ביטויים קלאסיים
מודול הביטויים הקלאסיים של Qiskit [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical) מכיל ייצוג חקרני של פעולות זמן ריצה על ערכים קלאסיים במהלך ביצוע מעגל. בשל מגבלות חומרה, רק תנאי `QuantumCircuit.if_test()` נתמכים כרגע.

הדוגמה הבאה מראה שניתן להשתמש בחישוב הזוגיות ליצירת מצב GHZ בן n-Qubit באמצעות מעגלים דינמיים. ראשית, צור $n/2$ זוגות Bell על Qubits סמוכים. לאחר מכן, חבר את הזוגות הללו יחד באמצעות שכבה של Gates CNOT בין הזוגות. לאחר מכן מודדים את ה-Qubit היעד של כל Gates CNOT הקודמים ומאפסים כל Qubit מדוד למצב $\vert 0 \rangle$. מיישמים $X$ על כל אתר שלא נמדד שעבורו זוגיות כל הסיביות הקודמות היא אי-זוגית. לבסוף, Gates CNOT מוחלים על Qubits המדודים כדי לשחזר את השזירה שאבדה במדידה.

בחישוב הזוגיות, האלמנט הראשון של הביטוי הנבנה כולל הרמת אובייקט Python `mr[0]` לצומת [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) (הפונקציה `lift` משמשת להפיכת אובייקטים שרירותיים לביטויים קלאסיים). אין צורך בכך עבור `mr[1]` ורגיסטרים הקלאסיים האפשריים הבאים, מכיוון שהם קלטים ל-`expr.bit_xor`, וכל הרמה נחוצה נעשית אוטומטית במקרים אלה. ניתן לבנות ביטויים כאלה בלולאות ובמבנים אחרים.

In [4]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [5]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg)

## מציאת Backends התומכים במעגלים דינמיים
כדי למצוא את כל ה-Backends שהחשבון שלך יכול לגשת אליהם ותומכים במעגלים דינמיים, הרץ קוד כמו הבא. דוגמה זו מניחה שכבר [שמרת את פרטי הכניסה שלך.](/guides/save-credentials) תוכל גם [לציין פרטי כניסה באופן מפורש](/guides/initialize-account#explicit) בעת אתחול חשבון שירות Qiskit Runtime שלך. זה יאפשר לך לצפות ב-Backends הזמינים על אינסטנס או סוג תוכנית ספציפי, לדוגמה.

> **Note:** - ה-Backends הזמינים לחשבון תלויים באינסטנס שצוין בפרטי הכניסה.
> - הגרסה החדשה של מעגלים דינמיים זמינה כעת לכל המשתמשים בכל ה-Backends. ראה [את ההכרזה](/announcements/product-updates/2025-09-25-new-dynamic-circuits) לפרטים נוספים.